# Groceries Dataset — Exploratory Data Analysis
**Objective:** Understand product frequency distributions, transaction sizes, seasonal trends, and prepare a clean binary transaction matrix for association rule mining.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

: 

## 1. Data Loading & Initial Inspection

In [2]:
df = pd.read_csv('Groceries_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Unique members: {df["Member_number"].nunique()}')
print(f'Unique items: {df["itemDescription"].nunique()}')
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'Groceries_dataset.csv'

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
df.dtypes

## 2. Data Cleaning & Preprocessing

In [ ]:
# Clean item descriptions
df['itemDescription'] = df['itemDescription'].str.strip().str.lower()

# Check for cancelled/invalid transactions
cancelled = df[df['itemDescription'].str.contains('cancel|return|error|void', case=False, na=False)]
print(f'Cancelled/invalid transactions: {len(cancelled)}')

# Parse dates
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df.dropna(subset=['Date'], inplace=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.month_name()
df['DayOfWeek'] = df['Date'].dt.day_name()

print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Clean dataset: {len(df)} rows, {df["itemDescription"].nunique()} unique items')

## 3. Product Frequency Analysis

In [ ]:
item_counts = df['itemDescription'].value_counts()
print('Item frequency statistics:')
print(item_counts.describe())

# Top 20 items
fig, ax = plt.subplots(figsize=(12, 7))
top20 = item_counts.head(20)
top20.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', 20)[::-1], edgecolor='white')
ax.set_xlabel('Purchase Count')
ax.set_title('Top 20 Most Purchased Items', fontweight='bold', fontsize=15)
ax.invert_yaxis()
for i, v in enumerate(top20.values):
    ax.text(v + 30, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Rare items
rare_items = item_counts[item_counts < 10]
print(f'Rare items (<10 purchases): {len(rare_items)} ({len(rare_items)/len(item_counts)*100:.1f}% of catalog)')

fig, ax = plt.subplots(figsize=(12, 7))
item_counts.tail(20).plot(kind='barh', ax=ax, color=sns.color_palette('magma', 20), edgecolor='white')
ax.set_xlabel('Purchase Count')
ax.set_title('20 Rarest Items in Catalog', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Frequency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(item_counts.values, bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Number of Purchases')
axes[0].set_ylabel('Number of Items')
axes[0].set_title('Item Frequency Distribution', fontweight='bold')
axes[0].axvline(item_counts.median(), color='red', linestyle='--', label=f'Median={item_counts.median():.0f}')
axes[0].legend()

axes[1].hist(item_counts.values, bins=50, color='#DD8452', edgecolor='white', alpha=0.85, log=True)
axes[1].set_xlabel('Number of Purchases')
axes[1].set_ylabel('Number of Items (log)')
axes[1].set_title('Item Frequency Distribution (Log Scale)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Transaction Size Analysis

In [ ]:
df['transaction_id'] = df['Member_number'].astype(str) + '_' + df['Date'].astype(str)
tx_sizes = df.groupby('transaction_id')['itemDescription'].count()

print(f'Total transactions: {len(tx_sizes)}')
print(tx_sizes.describe())

fig, ax = plt.subplots(figsize=(10, 6))
tx_sizes.clip(upper=20).hist(bins=range(1, 22), ax=ax, color='#55A868', edgecolor='white', rwidth=0.85)
ax.set_xlabel('Items per Transaction')
ax.set_ylabel('Number of Transactions')
ax.set_title('Transaction Size Distribution', fontweight='bold', fontsize=15)
ax.axvline(tx_sizes.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {tx_sizes.mean():.1f}')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 5. Seasonal & Temporal Trends

In [ ]:
# Monthly trend
fig, ax = plt.subplots(figsize=(14, 5))
monthly = df.groupby(df['Date'].dt.to_period('M')).size()
monthly.index = monthly.index.to_timestamp()
ax.plot(monthly.index, monthly.values, color='#4C72B0', linewidth=2, marker='o', markersize=4)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#4C72B0')
ax.set_title('Monthly Purchase Volume', fontweight='bold', fontsize=15)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Day of week & monthly seasonality
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df['DayOfWeek'].value_counts().reindex(day_order)
axes[0].bar(dow.index, dow.values, color=sns.color_palette('coolwarm', 7), edgecolor='white')
axes[0].set_title('Purchases by Day of Week', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
mc = df['Month'].value_counts().sort_index()
axes[1].bar(month_order, mc.values, color=sns.color_palette('Spectral', 12), edgecolor='white')
axes[1].set_title('Purchases by Month', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Word Cloud of Product Names

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
wc = WordCloud(width=1400, height=700, background_color='white', colormap='viridis',
               max_words=150, prefer_horizontal=0.7)
wc.generate_from_frequencies(item_counts.to_dict())
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Product Word Cloud (sized by frequency)', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.show()

## 7. Product Co-occurrence Heatmap

In [ ]:
top15 = item_counts.head(15).index.tolist()
df_t = df[df['itemDescription'].isin(top15)]

cooc = pd.DataFrame(0, index=top15, columns=top15)
for _, grp in df_t.groupby('transaction_id'):
    items = grp['itemDescription'].unique()
    if len(items) > 1:
        for i in range(len(items)):
            for j in range(i+1, len(items)):
                if items[i] in top15 and items[j] in top15:
                    cooc.loc[items[i], items[j]] += 1
                    cooc.loc[items[j], items[i]] += 1

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(cooc, dtype=bool), k=1)
sns.heatmap(cooc, mask=mask, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, square=True, ax=ax, annot_kws={'size': 9})
ax.set_title('Product Co-occurrence Matrix (Top 15)', fontweight='bold', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Binary Transaction Matrix (Basket Format)

In [ ]:
# Filter rare items
min_freq = 10
frequent = item_counts[item_counts >= min_freq].index
df_filt = df[df['itemDescription'].isin(frequent)]
print(f'Items after filter (>= {min_freq}): {len(frequent)}')

# Build binary basket
basket = df_filt.groupby(['transaction_id', 'itemDescription']).size().unstack(fill_value=0)
basket = (basket > 0).astype(int)
print(f'Transaction matrix: {basket.shape[0]} transactions × {basket.shape[1]} items')
print(f'Sparsity: {(1 - basket.sum().sum()/(basket.shape[0]*basket.shape[1]))*100:.2f}%')
basket.head()

In [ ]:
# Sparsity visualization
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(basket.iloc[:100, :50].values, cmap='Blues', aspect='auto', interpolation='none')
ax.set_xlabel('Items (first 50)')
ax.set_ylabel('Transactions (first 100)')
ax.set_title('Transaction Matrix Sparsity (100×50 sample)', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Save
basket.to_csv('transaction_matrix.csv')
print('Transaction matrix saved to transaction_matrix.csv')